# Build the Trip-Level Analytical Table

One row = one trip, enriched with cost and behaviour features. Spend = approved lines only.
Saves `trip_analytics.csv` to data/processed. This is the table SQL and Power BI use.

In [ ]:
import os
import numpy as np
import pandas as pd

PROC = r'c:\Users\Sweta\OneDrive\Desktop\proj2\data\processed'

employees = pd.read_csv(os.path.join(PROC, 'employees_clean.csv'))
trips = pd.read_csv(os.path.join(PROC, 'trips_clean.csv'), parse_dates=['booked_date','trip_start_date','trip_end_date'])
expenses = pd.read_csv(os.path.join(PROC, 'expenses_clean.csv'))
offices = pd.read_csv(os.path.join(PROC, 'offices_clean.csv'))
vendors = pd.read_csv(os.path.join(PROC, 'vendors_clean.csv'))
print('loaded clean files')

## 1. Cost per trip and per component (approved lines only)
Reason: total trip cost is not stored; we build it by summing approved expense amounts per trip.
A pivot gives one column per category so we can see flight vs hotel vs other.

In [ ]:
appr = expenses[expenses['is_approved'] == True].copy()

# total approved cost per trip
trip_cost = appr.groupby('trip_id')['amount'].sum().rename('total_cost')

# cost per category (pivot)
comp = appr.pivot_table(index='trip_id', columns='category', values='amount', aggfunc='sum', fill_value=0)
comp.columns = [f'{c}_cost' for c in comp.columns]

# policy exception count per trip (approved or not - exceptions are about policy, not payment)
exc = expenses.groupby('trip_id')['is_policy_exception'].sum().rename('policy_exception_count')

trip_money = pd.concat([trip_cost, comp, exc], axis=1).reset_index()
print('trips with money data:', len(trip_money))
trip_money.head()

## 2. Join trips + money + employee/office + vendor
Reason: bring behaviour (trip) and geography (office/region) and vendor status onto each trip.

In [ ]:
emp = employees[['employee_id','department','job_level','office_id']].merge(
    offices[['office_id','city','country','region']], on='office_id', how='left')

df = trips.merge(trip_money, on='trip_id', how='left')
df = df.merge(emp, on='employee_id', how='left')
df = df.merge(vendors[['vendor_id','vendor_name','is_preferred_vendor']], on='vendor_id', how='left')
print('joined shape:', df.shape)

## 3. Derived variables
- booking_lead_days = trip_start_date - booked_date (how early it was booked)
- trip_duration_days = trip_end_date - trip_start_date
- year / quarter from trip_start_date (for the trend)
- preferred_vendor_flag: preferred / non_preferred / unknown
- booking_window bucket (0-3, 4-7, 8-14, 15+ days)

In [ ]:
df['booking_lead_days'] = (df['trip_start_date'] - df['booked_date']).dt.days
df['trip_duration_days'] = (df['trip_end_date'] - df['trip_start_date']).dt.days
df['year'] = df['trip_start_date'].dt.year
df['quarter'] = df['trip_start_date'].dt.to_period('Q').astype('string')
df['month'] = df['trip_start_date'].dt.to_period('M').astype('string')

def vendor_flag(x):
    if x is True or x == True:
        return 'preferred'
    if x is False or x == False:
        return 'non_preferred'
    return 'unknown'
df['preferred_vendor_flag'] = df['is_preferred_vendor'].apply(vendor_flag)

def lead_bucket(d):
    if pd.isna(d):
        return 'unknown'
    if d <= 3:
        return '0-3 days'
    if d <= 7:
        return '4-7 days'
    if d <= 14:
        return '8-14 days'
    return '15+ days'
df['booking_window'] = df['booking_lead_days'].apply(lead_bucket)

df['has_policy_exception'] = df['policy_exception_count'].fillna(0) > 0
df[['trip_id','booking_lead_days','trip_duration_days','quarter','preferred_vendor_flag','booking_window']].head()

## 4. High-cost trip flag (simple percentile)
Reason: flag unusually expensive trips using the 90th percentile of completed-trip cost.
Simple and explainable - no statistics beyond a percentile.

In [ ]:
completed_cost = df.loc[(df['trip_status'] == 'completed') & (df['total_cost'].notna()), 'total_cost']
p90 = completed_cost.quantile(0.90)
print('90th percentile trip cost = %.2f' % p90)
df['is_high_cost'] = df['total_cost'] > p90

## 5. Save the analytical table
We save ALL trips, plus a completed+has-cost view used for cost-per-trip analysis.

In [ ]:
for c in ['flight_cost','hotel_cost','meals_cost','ground_transport_cost','misc_cost',
          'total_cost','policy_exception_count']:
    if c not in df.columns:
        df[c] = 0
    df[c] = df[c].fillna(0)

df.to_csv(os.path.join(PROC, 'trip_analytics.csv'), index=False)
print('saved trip_analytics.csv  shape:', df.shape)

analysis = df[(df['trip_status'] == 'completed') & (df['total_cost'] > 0)].copy()
analysis.to_csv(os.path.join(PROC, 'trip_analytics_completed.csv'), index=False)
print('saved trip_analytics_completed.csv  rows:', len(analysis))

## 6. Quick sanity peek (the answer starts here)
Average cost per trip by quarter, and component share. These are early signals for the analysis.

In [ ]:
print('AVG COST PER TRIP BY QUARTER (completed):')
print(analysis.groupby('quarter')['total_cost'].agg(['count','mean']).round(0).to_string())
print()
print('COMPONENT SHARE OF TOTAL SPEND:')
comp_tot = analysis[['flight_cost','hotel_cost','meals_cost','ground_transport_cost','misc_cost']].sum()
print((comp_tot / comp_tot.sum() * 100).round(1).to_string())